Author: Krish

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

### User input required
Put the data path on your system in the cell below

In [2]:
data_path = "CAR_-_EP_Flow_Activity_Queue__Agent_Names"

### User input ends

### Reading all filenames in the data folder

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


### Reading all data files
The code chunk below reads and appends all the CAR data files. The first two rows of each file are blank and thus ignored.

In [4]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)

1 CAR - EP, Flow, Activity, Queue, & Agent Names (01-12-25 - 01-18-25) (52010, 7)
2 CAR - EP, Flow, Activity, Queue, & Agent Names (01-19-25 - 02-01-25) (95495, 7)
3 CAR - EP, Flow, Activity, Queue, & Agent Names (02-02-25 - 02-15-25) (90056, 7)
4 CAR - EP, Flow, Activity, Queue, & Agent Names (02-16-25 - 03-01-25) (88186, 7)
5 CAR - EP, Flow, Activity, Queue, & Agent Names (03-02-25 - 03-15-25) (86377, 7)
6 CAR - EP, Flow, Activity, Queue, & Agent Names (04-07-24 - 04-20-24) (88766, 7)
7 CAR - EP, Flow, Activity, Queue, & Agent Names (04-21-24 - 05-04-24) (89643, 7)
8 CAR - EP, Flow, Activity, Queue, & Agent Names (05-05-24 - 05-18-24) (82575, 7)
9 CAR - EP, Flow, Activity, Queue, & Agent Names (05-19-24 - 06-01-24) (71103, 7)
10 CAR - EP, Flow, Activity, Queue, & Agent Names (06-02-24 - 06-15-24) (84354, 7)
11 CAR - EP, Flow, Activity, Queue, & Agent Names (06-16-24 - 06-29-24) (82124, 7)
12 CAR - EP, Flow, Activity, Queue, & Agent Names (06-30-24 - 07-13-24) (79752, 7)
13 CAR - EP, 

### Time datatype conversion
The code chunk below converts time from string to datetime datatype.

In [5]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [6]:
# Checking the datatype of all columns
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [7]:
# Creating a new column 'hour' as it will be useful to visualize peak calling hours
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

### Cleaning Data

In [8]:
# --- 1) FILTER ROWS ---------------------------------------------------------

# Define the three conditions

# Rows that represent a particular menu
cond1 = (
    df_main['EP Name'].notna() &
    df_main['Flow Name'].isna() &
    df_main['Activity Name'].isna()
)

# Rows that represent the beginning / end of a call journey
cond2 = (
    df_main['EP Name'].notna() &
    df_main['Activity Name'].notna() &
    df_main['Flow Name'].isna()
)

# rows that represent presence within a queue
cond3 = df_main['Queue Name'].notna()

# Keep rows that satisfy ANY of the three conditions
df_filt = df_main[cond1 | cond2 | cond3].copy()

# --- 2) TIME DELTA (seconds since previous row within each Contact Session ID) ---

# Ensure timestamp is datetime (safe if it's already datetime64[ns])
df_filt['Activity Start Timestamp'] = pd.to_datetime(df_filt['Activity Start Timestamp'], errors='coerce')

# Sort so "previous row" is truly the prior event in time for that session
# df_filt = df_filt.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

# Compute per-session time difference (first row per session will be NaN)
df_filt['seconds_since_prev'] = (
    df_filt.groupby('Contact Session ID')['Activity Start Timestamp']
           .diff()
           .dt.total_seconds()
)

# b) seconds until next row (last row per session -> NaN)
df_filt['seconds_until_next'] = (
    df_filt.groupby('Contact Session ID')['Activity Start Timestamp']
           .shift(-1)
           .sub(df_filt['Activity Start Timestamp'])
           .dt.total_seconds()
)

# df_filt now has the filtered rows plus the new 'seconds_since_prev' column

### Filtering 1: Filter out intake outdial calls

In [13]:
# Find all Contact Session IDs that contain "Intake Outdial EP"
outdial_sessions = df_filt.loc[df_filt["EP Name"] == "Intake Outdial EP", "Contact Session ID"].unique()

# Exclude those sessions from df_filt
df_outbound = df_filt[~df_filt["Contact Session ID"].isin(outdial_sessions)].copy()

### Filtering 2: Filter for calls that reached MainMenu

In [16]:
df_reached_main = df_outbound[df_outbound["Contact Session ID"].isin(
    df_outbound.loc[df_outbound["Activity Name"] == "MainMenu", "Contact Session ID"].unique()
)].copy()

num_reached_main = df_reached_main["Contact Session ID"].nunique() / df_outbound["Contact Session ID"].nunique()
print(num_reached_main)

0.6930854601814705


### Filtering 3: Filter for calls that reached beyond the MainMenu

In [ ]:
qualifying_sessions = []

for contact_id, group in df_reached_main.groupby("Contact Session ID"):
    # Find the first occurrence of MainMenu
    mainmenu_rows = group[group["Activity Name"] == "MainMenu"]
    if mainmenu_rows.empty:
        continue
    
    first_mainmenu_time = mainmenu_rows["Activity Start Timestamp"].iloc[0]
    
    # Rows that occur *after* the first MainMenu
    after_mainmenu = group[group["Activity Start Timestamp"] > first_mainmenu_time]
    
    # Check if there is at least one non-empty Activity Name that isn't MainMenu
    if after_mainmenu["Activity Name"].dropna().ne("MainMenu").any():
        qualifying_sessions.append(contact_id)

# Filter df_reached_main to only those sessions
df_reached_beyond_main = df_reached_main[df_reached_main["Contact Session ID"].isin(qualifying_sessions)].copy()

In [20]:
num_qual = df_reached_beyond_main["Contact Session ID"].nunique() / df_reached_main["Contact Session ID"].nunique()
print(num_qual)

0.9302812650893288


#### What percentage of calls were abandoned at MainMenu after prompt repetition (call never went beyond MainMenu)?

In [23]:
stopped_at_main_sessions = set(df_reached_main["Contact Session ID"]) - set(df_reached_beyond_main["Contact Session ID"])

# Filter df_reached_main for those sessions
df_stopped_at_main = df_reached_main[df_reached_main["Contact Session ID"].isin(stopped_at_main_sessions)]

# Count MainMenu occurrences per session
mainmenu_counts = (
    df_stopped_at_main[df_stopped_at_main["Activity Name"] == "MainMenu"]
    .groupby("Contact Session ID")
    .size()
)

# Calculate percentage of sessions with more than one MainMenu
num_multiple_mainmenu = (mainmenu_counts > 1).sum()
total_stopped_sessions = len(mainmenu_counts)
percentage_multiple_mainmenu = (num_multiple_mainmenu / total_stopped_sessions) * 100

print(f"Percentage of sessions with more than one MainMenu (but no activity beyond MainMenu): {percentage_multiple_mainmenu:.2f}%")

Percentage of sessions with more than one MainMenu (but no activity beyond MainMenu): 62.81%


### Filtering 4: Check if caller returns to MainMenu

In [ ]:
df_return_sessions = []

# Loop through each session
for contact_id, group in df_reached_beyond_main.groupby("Contact Session ID"):
    # Get activity names in order (dropping NaN)
    activities = group["Activity Name"].dropna().tolist()

    # Find indexes of all MainMenu occurrences
    mainmenu_indices = [i for i, a in enumerate(activities) if a == "MainMenu"]

    # Check if there are non-consecutive MainMenu entries
    if len(mainmenu_indices) > 1:
        non_consecutive = any(mainmenu_indices[i+1] - mainmenu_indices[i] != 1 
                              for i in range(len(mainmenu_indices) - 1))
        if non_consecutive:
            df_return_sessions.append(contact_id)

# Create DataFrames
df_return = df_reached_beyond_main[df_reached_beyond_main["Contact Session ID"].isin(df_return_sessions)].copy()
df_no_return = df_reached_beyond_main[~df_reached_beyond_main["Contact Session ID"].isin(df_return_sessions)].copy()


In [27]:
num_no_return = df_no_return["Contact Session ID"].nunique() / df_reached_beyond_main["Contact Session ID"].nunique()
num_return = df_return["Contact Session ID"].nunique() / df_reached_beyond_main["Contact Session ID"].nunique()
print(num_no_return)
print(num_return)

0.9499185747004133
0.05008142529958671


In [30]:
# Count MainMenu occurrences per session
mainmenu_counts = (
    df_no_return[df_no_return["Activity Name"] == "MainMenu"]
    .groupby("Contact Session ID")
    .size()
)

# Number of sessions with more than 1 MainMenu
num_multiple_mainmenu = (mainmenu_counts > 1).sum()

# Total number of sessions in df_no_return
total_sessions = len(mainmenu_counts)

# Percentage
percentage_multiple_mainmenu = (num_multiple_mainmenu / total_sessions) * 100

print(f"Percentage of df_no_return sessions with multiple MainMenu instances: {percentage_multiple_mainmenu:.2f}%")

Percentage of df_no_return sessions with multiple MainMenu instances: 1.53%


### Filtering 5: Check if caller reached a queue

In [ ]:
# Helper function to get success vs abandoned sessions
def split_success_abandoned(df):
    # Find session IDs with success
    success_sessions = df.loc[
        (df["Activity Name"] == "ClosedQueueMenu") | (df["Queue Name"].notna()), 
        "Contact Session ID"
    ].unique()
    
    # Successful calls
    df_success = df[df["Contact Session ID"].isin(success_sessions)].copy()
    
    # Abandoned calls
    df_abandoned = df[~df["Contact Session ID"].isin(success_sessions)].copy()
    
    return df_success, df_abandoned

df_return_success, df_return_abandoned = split_success_abandoned(df_return)
df_no_return_success, df_no_return_abandoned = split_success_abandoned(df_no_return)

In [29]:
num_returns = df_return_success["Contact Session ID"].nunique() / df_return["Contact Session ID"].nunique()
num_no_returns = df_no_return_success["Contact Session ID"].nunique() / df_no_return["Contact Session ID"].nunique()
print(num_returns)
print(num_no_returns)

0.6542298225158699
0.7575233932108463


In [ ]:
# Total unique sessions
total_sessions = df_no_return_success["Contact Session ID"].nunique()

# Sessions with at least one non-NA Queue Name
sessions_with_queue = df_no_return_success.loc[
    df_no_return_success["Queue Name"].notna(), "Contact Session ID"
].nunique()

# Percentage
percentage_with_queue = (sessions_with_queue / total_sessions) * 100

print(f"Percentage of df_no_return_success calls with non-NA Queue Name: {percentage_with_queue:.2f}%")

Percentage of df_no_return_success calls with non-NA Queue Name: 65.98%


In [ ]:
# Total unique sessions
total_sessions = df_return_success["Contact Session ID"].nunique()

# Sessions with at least one non-NA Queue Name
sessions_with_queue = df_return_success.loc[
    df_return_success["Queue Name"].notna(), "Contact Session ID"
].nunique()

# Percentage
percentage_with_queue = (sessions_with_queue / total_sessions) * 100

print(f"Percentage of df_return_success calls with non-NA Queue Name: {percentage_with_queue:.2f}%")

Percentage of df_return_success calls with non-NA Queue Name: 82.02%


### Analysis of next activity from Main Menu

In [ ]:
from collections import Counter

next_activities = []

# Loop through each session
for contact_id, group in df_reached_beyond_main.groupby("Contact Session ID"):
    
    # Get activity names
    activities = group["Activity Name"].dropna().tolist()
    
    # Find indexes of all MainMenu occurrences
    mainmenu_indices = [i for i, a in enumerate(activities) if a == "MainMenu"]
    
    if mainmenu_indices:
        last_mainmenu_idx = mainmenu_indices[-1]
        # Check if there is a next activity after the last MainMenu
        if last_mainmenu_idx + 1 < len(activities):
            next_activities.append(activities[last_mainmenu_idx + 1])

# Count occurrences of each next activity
next_activity_counts = Counter(next_activities)

# Convert to DataFrame for counts and percentages
next_activity_df = pd.DataFrame.from_dict(next_activity_counts, orient="index", columns=["Count"])
next_activity_df.index.name = "Next Activity"
next_activity_df["Percentage"] = (next_activity_df["Count"] / next_activity_df["Count"].sum()) * 100

next_activity_df = next_activity_df.sort_values("Count", ascending=False)

print(next_activity_df)

                                Count  Percentage
Next Activity                                    
SeniorsMenu                     87156   57.130496
StaffDirectoryEnglishTransfer   25938   17.002281
ClinicVoicemailTransfer         12940    8.482131
AppointmentMenu                 11264    7.383518
HelpWithLegalorOtherReasonMenu   8762    5.743465
StaffDirectorySpanishTransfer    3386    2.219513
ClosedMenu                       2031    1.331314
FrontDeskTransfer3                541    0.354624
ComplimentOrComplaintMenu         380    0.249089
AddressFaxHoursMenu               154    0.100947
LegalMenu2                          4    0.002622


#### Analysis of instances of tranferring from ComplimentOrComplaintMenu --> Front Desk --> Main Menu (Done after class presentation)

In [39]:
df_complaint = df_reached_beyond_main.groupby('Contact Session ID').filter(
    lambda x: (x['Activity Name'] == 'ComplimentOrComplaintMenu').any()
)

In [ ]:
def has_mainmenu_after_frontdesk(group):
    idxs_frontdesk = group.index[group['Activity Name'] == 'FrontDeskTransfer2'].tolist()
    if not idxs_frontdesk:
        return False 
    first_fd_idx = idxs_frontdesk[0]
    
    # Check if any MainMenu comes after
    idxs_mainmenu = group.index[group['Activity Name'] == 'MainMenu'].tolist()
    return any(i > first_fd_idx for i in idxs_mainmenu)

# Apply to each session and count
calls_with_mainmenu_after_frontdesk = (
    df_complaint.groupby('Contact Session ID').filter(has_mainmenu_after_frontdesk)['Contact Session ID'].nunique()
)

print("Number of calls with MainMenu after first FrontDeskTransfer2:", calls_with_mainmenu_after_frontdesk)

Number of calls with MainMenu after first FrontDeskTransfer2: 0
